# Week 5-2 — Re-ranking (Cross-Encoder)

**목적**: 로우레벨 검증에서 확인한 e5의 약한 변별력(무관 문장도 유사도 0.79, HER2 양성/음성 구분 실패)을 **Cross-Encoder 재정렬로 보정**한다. 1차 검색(Hybrid, 5-1 채택)으로 top-20을 넉넉히 가져온 뒤, reranker가 질문-chunk 쌍을 정밀 채점해 top-5로 재정렬한다.

- **1차 검색**: 5-1에서 채택한 Hybrid(BM25+Dense, RRF) — top-20
- **Reranker**: BAAI/bge-reranker-v2-m3 (다국어, 한국어 강함) — sentence-transformers CrossEncoder로 로드
- **비용 절약**: R1(Dense)/R2(BM25)/R3(Hybrid)는 5-1 CSV를 재사용하고, **R4(Hybrid+Rerank)만 신규 채점**. 최종적으로 과제의 4구성 ablation 표를 완성한다.
- judge: gpt-4o-mini (5주차 통일).

---
## 1. 설정 + 전처리 상수 (5-0/5-1과 동일)

In [1]:
from pathlib import Path
import os, json, re
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"

load_dotenv(PROJECT_ROOT / ".env"); load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 없음"

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
GEN_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o-mini"
TOP_K = 5
FETCH_K = 20   # 1차 검색 폭 (rerank 대상)
RRF_K = 60

CHUNK_SIZE_BY_LANG    = {"ko": 540, "en": 620, "unknown": 580}
CHUNK_OVERLAP_BY_LANG = {"ko": 80,  "en": 90,  "unknown": 85}

CUT_FROM_PAGE = {
    "esmo_breast_cancer_patient_guide_korean.pdf": 60,
    "ncc_breast_cancer_screening_guideline_2015.pdf": 109,
    "nccn_metastatic_breast_cancer_patient.pdf": 63,
}
KBCS_REF_PAGES = set([
    109, 111, 115, 117, 121, 123, 124, 125, 126, 127, 129, 130, 131, 132, 135,
    137, 139, 141, 172, 173, 174, 223, 232, 233, 234, 235, 236, 237, 238, 240,
    241, 242, 243, 244, 245, 247, 248, 249, 250, 251, 252,
    50, 51, 53, 56, 100, 101, 103, 104, 108, 110, 112, 113, 114, 116, 118,
    119, 120, 122, 128, 133, 134, 136, 138, 140, 163, 164, 167, 168, 169, 170,
    171, 221, 222, 224, 227, 228, 229, 230, 231, 239, 246,
])
KBCS_NAME = "kbcs_korean_breast_cancer_guideline_2023.pdf"
print("설정 완료 | reranker:", RERANK_MODEL)

설정 완료 | reranker: BAAI/bge-reranker-v2-m3


---
## 2. P1 chunk + Dense + BM25 + Hybrid 재구성 (5-1과 동일)

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

manifest_path = DATA_RAW / "metadata" / "manifest.json"
meta_lookup = {}
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        meta_lookup = {m["filename"]: m for m in json.load(f) if m.get("downloaded")}

def guess_lang(text):
    return "ko" if len(re.findall(r"[\uac00-\ud7a3]", text)) > 20 else "en"

def load_docs_p1():
    out = []
    for pdf in sorted((DATA_RAW / "pdf").rglob("*.pdf")):
        cut = CUT_FROM_PAGE.get(pdf.name)
        for d in PyMuPDFLoader(str(pdf)).load():
            pg = d.metadata.get("page", 0)
            if cut is not None and pg >= cut: continue
            if pdf.name == KBCS_NAME and pg in KBCS_REF_PAGES: continue
            extra = meta_lookup.get(pdf.name, {})
            d.metadata.update({"filename": pdf.name, "org": extra.get("org", pdf.parent.name),
                "title": extra.get("title", pdf.stem),
                "language": extra.get("language", guess_lang(d.page_content)), "page": pg})
            out.append(d)
    return out

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]
def split_docs(docs):
    out = []
    for lang in set(d.metadata.get("language", "unknown") for d in docs):
        size = CHUNK_SIZE_BY_LANG.get(lang, 580); ov = CHUNK_OVERLAP_BY_LANG.get(lang, 85)
        sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=ov,
                                            separators=SEPARATORS, length_function=len)
        out.extend(sp.split_documents([d for d in docs if d.metadata.get("language") == lang]))
    return out

chunks = split_docs(load_docs_p1())
print(f"P1 chunk: {len(chunks)}개 (2,240 기대)")

P1 chunk: 2240개 (2,240 기대)


In [3]:
import os as _os
from huggingface_hub import snapshot_download
_os.environ.pop("HF_HUB_OFFLINE", None); _os.environ.pop("TRANSFORMERS_OFFLINE", None)
_os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

try:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=True)
except Exception:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=False, max_workers=1)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

embeddings = HuggingFaceEmbeddings(model_name=model_dir,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16})

vdir = VECTOR_ROOT / "week5pre_P1_ref_removed"
coll = "breast_rag_week5pre_P1_ref_removed"
vs = Chroma(collection_name=coll, embedding_function=embeddings, persist_directory=str(vdir))
assert vs._collection.count() > 0, "P1 컬렉션 없음 — 5-0 먼저 실행"
print(f"Dense(P1) 재사용: {vs._collection.count()}개")

def dense_search(query, k=TOP_K):
    return vs.similarity_search(query, k=k)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Dense(P1) 재사용: 2240개


In [4]:
from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi
from tqdm import tqdm

kiwi = Kiwi()
def tokenize(text):
    return [t.form.lower() for t in kiwi.tokenize(text)
            if t.tag[0] in ("N", "V", "S") or t.tag in ("SL", "SN", "XR")]

corpus_tokens = [tokenize(c.page_content) for c in tqdm(chunks, desc="BM25 토큰화")]
bm25 = BM25Okapi(corpus_tokens)

def bm25_search(query, k=TOP_K):
    scores = bm25.get_scores(tokenize(query))
    top_idx = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]
    return [chunks[i] for i in top_idx]

def _key(doc):
    return (doc.metadata.get("filename"), doc.metadata.get("page"), doc.page_content[:80])

def hybrid_search(query, k=TOP_K, fetch_k=FETCH_K):
    dense_docs = dense_search(query, k=fetch_k)
    bm25_docs = bm25_search(query, k=fetch_k)
    scores, registry = {}, {}
    for rank, d in enumerate(dense_docs):
        kk = _key(d); registry[kk] = d
        scores[kk] = scores.get(kk, 0) + 1.0 / (RRF_K + rank + 1)
    for rank, d in enumerate(bm25_docs):
        kk = _key(d); registry[kk] = d
        scores[kk] = scores.get(kk, 0) + 1.0 / (RRF_K + rank + 1)
    top = sorted(scores, key=lambda kk: -scores[kk])[:k]
    return [registry[kk] for kk in top]

print("Dense / BM25 / Hybrid 준비 완료")

BM25 토큰화: 100%|██████████| 2240/2240 [00:04<00:00, 476.66it/s] 

Dense / BM25 / Hybrid 준비 완료


---
## 3. Reranker 로드 + R4 정의

bge-reranker-v2-m3는 질문-문서 쌍을 함께 입력받는 **Cross-Encoder** — bi-encoder(e5)보다 정밀한 관련도 판정. 첫 실행 시 모델 다운로드(~2.3GB) 필요. CPU 추론이라 문항당 수 초.

In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANK_MODEL, max_length=512, device="cpu")
print("reranker 로드 완료:", RERANK_MODEL)

def hybrid_rerank_search(query, k=TOP_K, fetch_k=FETCH_K):
    """Hybrid top-20 -> Cross-Encoder 재정렬 -> top-5"""
    candidates = hybrid_search(query, k=fetch_k, fetch_k=fetch_k)
    pairs = [(query, d.page_content) for d in candidates]
    scores = reranker.predict(pairs)
    order = sorted(range(len(candidates)), key=lambda i: -scores[i])
    return [candidates[i] for i in order[:k]]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

reranker 로드 완료: BAAI/bge-reranker-v2-m3


---
## 4. 로우레벨 sanity check (API 0원)

Rerank 전(Hybrid 순위) vs 후(재정렬 순위)를 나란히 출력 — reranker가 실제로 순위를 바꾸는지, 바꾼 방향이 타당한지 눈으로 확인.

In [6]:
test_queries = [
    "HER2 양성 유방암은 어떤 표적 치료를 하나요?",
    "유방암 1기와 2기의 차이는 무엇인가요?",
]
for q in test_queries:
    print("=" * 70)
    print(f"Q: {q}")
    before = hybrid_search(q, k=5)
    after = hybrid_rerank_search(q, k=5)
    print("\n  [Rerank 전] Hybrid top-5")
    for i, d in enumerate(before, 1):
        print(f"    {i}. [{d.metadata.get('filename','?')[:20]} p.{d.metadata.get('page','?')}] "
              f"{d.page_content.strip()[:70].replace(chr(10),' ')}")
    print("\n  [Rerank 후] top-5")
    for i, d in enumerate(after, 1):
        moved = "" if i <= len(before) and _key(d) == _key(before[i-1]) else "  <- 순위 변경"
        print(f"    {i}. [{d.metadata.get('filename','?')[:20]} p.{d.metadata.get('page','?')}] "
              f"{d.page_content.strip()[:70].replace(chr(10),' ')}{moved}")
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: HER2 양성 유방암은 어떤 표적 치료를 하나요?

  [Rerank 전] Hybrid top-5
    1. [kbcs_korean_breast_c p.72] 2023 The 10 th Korean Clinical Practice Guideline for Breast Cancer | 
    2. [kbcs_korean_breast_c p.149] p=0.0063). 하지만 3제 병용요법의 우수한 효과만큼 부작용도 심하였고, 특히 3등급 이상의 설사가 약 13%의 환 자에
    3. [esmo_breast_cancer_p p.7] 중요합니다.  HER2 는 세포의 성장에 관여하는 또다른 수용체이며, 유방암의 약 20% 에 존재합니다. HER2  발현이가 
    4. [kbcs_korean_breast_c p.148] 킨다는 증거는 있지만 충분하지 않고, 전체 생존기간의 차이는 미약하기 때문에 항암화학요법 의 기간은 부작용과 전반적인 삶의 질
    5. [esmo_breast_cancer_p p.23] 24 유방암 표적 치료 표적 요법은 암세포의 성장을 촉진하는 특정 신호 경로를 차단하는 약물을 사용하는 치료법입니다.  유방암

  [Rerank 후] top-5
    1. [kbcs_korean_breast_c p.72] 2023 The 10 th Korean Clinical Practice Guideline for Breast Cancer | 
    2. [esmo_breast_cancer_p p.23] 24 유방암 표적 치료 표적 요법은 암세포의 성장을 촉진하는 특정 신호 경로를 차단하는 약물을 사용하는 치료법입니다.  유방암  <- 순위 변경
    3. [kbcs_korean_breast_c p.149] 150 |  2023 제10차 한국유방암 진료권고안 150 |  2023 제10차 한국유방암 진료권고안 경우, HER2를 표적  <- 순위 변경
    4. [esmo_breast_cancer_p p.7] 중요합니다.  HER2 는 세포의

### sanity 관찰 메모 (직접 채우기)
- reranker가 순위를 실제로 바꾸는가? 위로 올라온 chunk가 더 관련 있어 보이는가?

---
## 5. R4만 신규 채점 → 4구성 ablation 완성

R1~R3는 5-1 CSV 재사용 (비용 0). R4(Hybrid+Rerank)만 생성+채점한다.

In [7]:
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)
RAG_PROMPT = ChatPromptTemplate.from_template(
    """당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]""")

def format_context(docs):
    return "\n\n---\n\n".join(
        f"[{i}] 출처: {d.metadata.get('org','?')} / {d.metadata.get('title','?')} / p.{d.metadata.get('page','?')}\n{d.page_content}"
        for i, d in enumerate(docs, 1))

golden = pd.read_csv(DATA_EVAL / "golden_set_v1.csv").to_dict("records")
print(f"golden_set: {len(golden)}문항")

golden_set: 30문항


In [8]:
import nest_asyncio; nest_asyncio.apply()
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]

rows = []
for g in tqdm(golden, desc="RAG[R4_hybrid_rerank]"):
    docs = hybrid_rerank_search(g["question"])
    ans = (llm | StrOutputParser()).invoke(
        RAG_PROMPT.format(context=format_context(docs), question=g["question"]))
    rows.append({"user_input": g["question"], "response": ans,
                 "retrieved_contexts": [d.page_content for d in docs],
                 "reference": g.get("ground_truth", "")})
ds = EvaluationDataset.from_list(rows)
df_r4 = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb).to_pandas()
df_r4.to_csv(DATA_PROCESSED / "week5_ragas_R4_hybrid_rerank.csv", index=False, encoding="utf-8-sig")
print("R4 완료")

RAG[R4_hybrid_rerank]:   0%|          | 0/30 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
RAG[R4_hybrid_rerank]: 100%|██████████| 30/30 [03:11<00:00,  6.39s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

R4 완료


In [9]:
# 4구성 ablation 표 (R1~R3는 5-1 CSV 로드)
score_tables = {
    "R1_dense": pd.read_csv(DATA_PROCESSED / "week5_ragas_R1_dense.csv"),
    "R2_bm25": pd.read_csv(DATA_PROCESSED / "week5_ragas_R2_bm25.csv"),
    "R3_hybrid": pd.read_csv(DATA_PROCESSED / "week5_ragas_R3_hybrid.csv"),
    "R4_hybrid_rerank": df_r4,
}

def _lang(q): return "KO" if re.search("[가-힣]", str(q)) else "EN"

rows = []
for name, df in score_tables.items():
    row = {"retriever": name}
    for c in METRIC_COLS:
        row[c] = round(df[c].mean(), 4)
    rows.append(row)
cmp = pd.DataFrame(rows)
for c in METRIC_COLS:
    cmp[c + "_delta"] = (cmp[c] - cmp[c].iloc[0]).round(4)
cmp.to_csv(DATA_PROCESSED / "week5_ablation_comparison.csv", index=False, encoding="utf-8-sig")
print(cmp.to_string())

print("\n--- 언어별 context_precision ---")
for L in ["KO", "EN"]:
    for name, df in score_tables.items():
        d2 = df.copy(); d2["_l"] = d2["user_input"].apply(_lang)
        sub = d2[d2["_l"] == L]
        print(f"  [{L}] {name:18s} CP={sub['context_precision'].mean():.4f}")
    print()

          retriever  faithfulness  answer_relevancy  context_precision  faithfulness_delta  answer_relevancy_delta  context_precision_delta
0          R1_dense        0.7487            0.6120             0.9072              0.0000                  0.0000                   0.0000
1           R2_bm25        0.6542            0.4482             0.8579             -0.0945                 -0.1638                  -0.0493
2         R3_hybrid        0.7784            0.6090             0.9378              0.0297                 -0.0030                   0.0306
3  R4_hybrid_rerank        0.8028            0.6105             0.9331              0.0541                 -0.0015                   0.0259

--- 언어별 context_precision ---
  [KO] R1_dense           CP=0.9602
  [KO] R2_bm25            CP=0.9040
  [KO] R3_hybrid          CP=0.9883
  [KO] R4_hybrid_rerank   CP=0.9815

  [EN] R1_dense           CP=0.8011
  [EN] R2_bm25            CP=0.7656
  [EN] R3_hybrid          CP=0.8367
  [EN] R4_hybrid

---
## 6. 문항별 분석 + Error Case (과제 6번 재료)

In [10]:
# R4 vs R3: rerank가 무엇을 바꿨나
h_ = score_tables["R3_hybrid"][["user_input", "context_precision"]]
r_ = df_r4[["user_input", "context_precision"]]
m = h_.merge(r_, on="user_input", suffixes=("_hybrid", "_rerank"))
m["delta"] = (m["context_precision_rerank"] - m["context_precision_hybrid"]).round(3)
m["lang"] = m["user_input"].apply(_lang)

print("=== Rerank로 CP가 변한 문항 (|delta| > 0.05) ===")
for _, r in m[abs(m["delta"]) > 0.05].sort_values("delta", ascending=False).iterrows():
    print(f"  {r['delta']:+.2f} ({r['lang']}) hybrid={r['context_precision_hybrid']:.2f} -> rerank={r['context_precision_rerank']:.2f} | {str(r['user_input'])[:42]}")
print(f"\n개선 {(m['delta']>0.05).sum()} / 악화 {(m['delta']<-0.05).sum()} / 동일 {(abs(m['delta'])<=0.05).sum()}")

# Error Case: R4 기준 CP 최저 3문항 + 검색된 chunk 출력
print("\n" + "=" * 70)
print("=== Error Case: R4 CP 최저 3문항 (retrospective 재료) ===")
r4full = df_r4.copy(); r4full["lang"] = r4full["user_input"].apply(_lang)
import ast
for _, r in r4full.nsmallest(3, "context_precision").iterrows():
    print(f"\n[CP={r['context_precision']:.2f}] ({r['lang']}) {r['user_input']}")
    ctxs = r["retrieved_contexts"]
    if isinstance(ctxs, str):
        try: ctxs = ast.literal_eval(ctxs)
        except Exception: ctxs = [ctxs]
    for i, c in enumerate(ctxs[:3], 1):
        print(f"    [{i}] {str(c)[:110].replace(chr(10), ' ')}")

=== Rerank로 CP가 변한 문항 (|delta| > 0.05) ===
  +0.08 (EN) hybrid=0.92 -> rerank=1.00 | What are the possible signs of inflammator
  +0.05 (EN) hybrid=0.75 -> rerank=0.80 | What factors put a person at increased ris
  -0.11 (EN) hybrid=0.92 -> rerank=0.80 | What are the different types of breast bio
  -0.11 (EN) hybrid=1.00 -> rerank=0.89 | When might a breast MRI be recommended for
  -0.32 (KO) hybrid=1.00 -> rerank=0.68 | 유방암 검진은 몇 살부터 받는 것이 권장되나요?

개선 2 / 악화 3 / 동일 25

=== Error Case: R4 CP 최저 3문항 (retrospective 재료) ===

[CP=0.00] (EN) What are the four BI-RADS breast density categories?
    [1] see how the cancer is responding to treatment. BI-RADS breast density BI-RADS breast density categories are  d
    [2] A radiologist will categorize your mammogram  results using a BI-RADS numbered system of  0 through 6 and clas
    [3] ㆍPPV3: 6.5% (3/46)in BI-RADS 4 ㆍPPV2: 5.6% (3/54) in BI-RADS 3-4 ㆍPPV3: 5.7% (3/53) in BI-RADS 3-4 Giuliano  (

[CP=0.68] (KO) 유방암 검진은 몇 살부터 받는 것이 권장되나요?
    [

---
## 판정 & 기록 (직접 채우기)

- **Rerank 효과**: Hybrid 대비 개선/악화 문항 수와 폭. 로우레벨에서 본 "e5 약한 변별력"이 실제로 보정됐나?
- **4구성 최종 순위**: Dense / BM25 / Hybrid / Hybrid+Rerank — CP·언어별 기준.
- **트레이드오프**: rerank 추가로 문항당 지연(초 단위) 발생 — 개선 폭이 이 비용을 정당화하나?
- **Error Case 3개**: 여전히 실패하는 문항과 원인 가설 (→ retrospective + 6주차 Agentic RAG 근거).
- **판정**: 최종 retrieval 구성 확정 — 5-3(Multi-Query)의 base로 무엇을 쓸지.
- decision_log에 한 줄 요약.